In [ ]:
# 准备训练、评测和播放录像的工具。
import lightning as L
import torch
from IPython.display import Video

from training.evaluate import evaluate, record_video
from training.ppo import PPO, ActorCritic, PPOData
from training.sim import SKILLS, DuckEnv, hardware


In [ ]:
# 用现行硬件并行跑平地直走任务。
hw = hardware.load("2026-11-26-hd1910")
skill = SKILLS["course-walk"]
env = DuckEnv(skill, hw, num_envs=2048, device="cuda", seed=1)
env.reset()
env.randomize_episode_starts()                           # 回合进度打散，免得 2048 只在同一步一起超时
# 按环境的观测和动作维度建策略网络与价值网络。
net = ActorCritic(env.obs_dim, env.critic_obs_dim, env.action_dim).to("cuda")
net

In [ ]:
data = PPOData(env, net)      # data：每轮用当前网络采 24 步 → 算优势 → 切成 4 个小批、用 5 遍
model = PPO(net)              # model：一个小批的损失，更新网络

In [ ]:
# 先采一轮，看看 PPO 的小批里有哪些训练数据。
batches = data.batches(data.prepare(data.collect()))
len(batches), {key: tuple(value.shape) for key, value in batches[0].items()}

In [ ]:
# 先记下没训练的网络会摔几次、每步平均拿多少奖励。
before = evaluate(net, skill, hw,
                  common_step_counter=env.unwrapped.common_step_counter)   # 评测对齐训练走到的课程档
before["falls"], round(before["mean_step_reward"], 3)

In [ ]:
# 演示只训 300 轮，用来比较训练前后的变化。
# 每轮重建一次数据，也就是每轮用最新的网络重新采样
trainer = L.Trainer(max_epochs=300, reload_dataloaders_every_n_epochs=1,
                    accelerator="gpu", devices=1, logger=False,
                    enable_checkpointing=False, enable_model_summary=False)
trainer.fit(model, data)

In [ ]:
# 训完用同一任务、同样的指标再评测一次。
after = evaluate(net, skill, hw,
                 common_step_counter=env.unwrapped.common_step_counter)
after["falls"], round(after["mean_step_reward"], 3)

In [ ]:
# 固定前进指令，录下训练后的走路表现。
record_video(net, skill, hw, "m04_walk.mp4",
             common_step_counter=env.unwrapped.common_step_counter)
Video("m04_walk.mp4", embed=True)

In [ ]:
# 留下演示权重，之后可直接载入，不用重训。
torch.save(net.state_dict(), "m04_walk.pt")